<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Contar, Decidir y Resistir: Más Allá de la Línea Recta 📈
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 06
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/06%20-%20Modelamiento%20Estadistico/Para%20Dummies/01_Modelos_GLM_Robustos_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 01 del Módulo 06 (Modelos lineales generalizados y métodos no paramétricos)](../01_Modelos_Lineales_Generalizados_y_No_Parametricos.ipynb). La **línea recta** es la herramienta más famosa de la estadística, pero tiene mañas: puede predecir **plagas negativas**, **probabilidades de 130 %** y dejarse arrastrar por **un solo vendedor tramposo**. Hay primos de la línea recta que evitan cada problema.

| En la plaza y el campo | En estadística |
|---|---|
| Contar plagas en una parcela (0, 1, 2, ... nunca negativo) | Modelo para **conteos** (*Poisson*) |
| «¿Se enfermó el cultivo? Sí o no» | Modelo para **sí/no** (*regresión logística*) |
| Cada punto más de humedad **multiplica** las plagas por 2.2 | Efecto **multiplicativo** (se interpreta como «veces») |
| Un vendedor que cobra una barbaridad por un bulto | Un **dato atípico** |
| Preguntar «¿cuánto cuesta lo típico?» en lugar de «el promedio» | **Mediana** y **línea de la mediana** (robusta) |

Al terminar podrás explicar:
1. Por qué la recta puede predecir **plagas negativas** y cómo se evita.
2. Por qué para un **sí/no** conviene una curva en «S» y no una recta.
3. Por qué **un dato loco** arrastra al promedio y a la recta, pero no a la mediana.

> 🔗 Si luego quieres ver las versiones **con fórmulas, el algoritmo paso a paso y simulaciones**: [cuaderno estándar](../01_Modelos_Lineales_Generalizados_y_No_Parametricos.ipynb). Taller del módulo: [`../../homeworks/Para Dummies/06_Modelamiento_Estadistico_Hands_On_Dummies.ipynb`](../../homeworks/Para%20Dummies/06_Modelamiento_Estadistico_Hands_On_Dummies.ipynb).

---
## 1. Plagas Negativas: la Recta Se Pasa de la Raya 🐛

Contamos plagas en 150 parcelas **inventadas** según la humedad. La verdad es que las plagas **se multiplican** con la humedad (cada punto más de humedad las multiplica por 2.2 aproximadamente). Una recta no sabe multiplicar: en las parcelas secas sigue bajando y llega a predecir **menos de cero plagas**. El modelo para conteos (*Poisson*) predice siempre **valores positivos**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, PoissonRegressor, LogisticRegression, QuantileRegressor

rng = np.random.default_rng(7)

# 150 parcelas INVENTADAS: las plagas se MULTIPLICAN con la humedad (-2 = muy seca, +2 = muy humeda)
humedad = rng.uniform(-2, 2, 150)
plagas = rng.poisson(np.exp(0.7 + 0.8 * humedad))
X = humedad.reshape(-1, 1)

recta = LinearRegression().fit(X, plagas)
poisson = PoissonRegressor(alpha=0).fit(X, plagas)
malla = np.linspace(-2, 2, 100).reshape(-1, 1)

negativas = int((recta.predict(malla) < 0).sum())
print(f"La recta predice menos de cero plagas en {negativas} de 100 puntos de la malla; el modelo de conteos, en {int((poisson.predict(malla) < 0).sum())}.")
print(f"Cada punto mas de humedad MULTIPLICA las plagas por {np.exp(poisson.coef_[0]):.2f} (verdad: {np.exp(0.8):.2f})")
assert negativas > 0 and (poisson.predict(malla) > 0).all() and abs(np.exp(poisson.coef_[0]) - np.exp(0.8)) < 0.3

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(humedad, plagas, s=12, alpha=0.5, color="#64748b")
ax.plot(malla, recta.predict(malla), color="#dc2626", lw=2, label="recta")
ax.plot(malla, poisson.predict(malla), color="#f59e0b", lw=2, label="modelo de conteos")
ax.axhline(0, color="k", lw=0.8); ax.set(xlabel="humedad", ylabel="plagas por parcela", title="La recta se va a números negativos"); ax.legend()
plt.show()

---
**Lección:** en las parcelas más secas la recta predice **plagas negativas**, algo imposible. El modelo para conteos nunca baja de cero y, además, se lee fácil: «**cada punto más de humedad multiplica las plagas por ≈ 2.3**» (la verdad inventada es 2.23). Esta idea —modelar **el promedio** con una forma que respete lo que se mide— es la de los **modelos lineales generalizados**.

---
## 2. ¿Se Enfermó o No? La Curva en «S» 🍄

Ahora la respuesta es **sí o no** (¿apareció el hongo?) según la lluvia del mes. Una recta puede dar «probabilidades» **menores que 0 o mayores que 1**; la curva en «S» (*regresión logística*) siempre se queda entre 0 y 1: casi nunca con poca lluvia, casi siempre con mucha, y cambia rápido en la mitad.

In [ ]:
rng = np.random.default_rng(21)
lluvia = rng.normal(0, 1.5, 300)
p_verdadera = 1 / (1 + np.exp(-(-0.3 + 1.8 * lluvia)))                      # INVENTADA
hongo = rng.binomial(1, p_verdadera)
XL = lluvia.reshape(-1, 1)

recta_h = LinearRegression().fit(XL, hongo)
curva = LogisticRegression(C=1e6).fit(XL, hongo)

extremos = np.array([[-4.0], [0.0], [4.0]])
print("Lluvia (muy poca, normal, mucha):", extremos.ravel())
print("   'probabilidad' con la recta:  ", recta_h.predict(extremos).round(2))
print("   probabilidad con la curva S:  ", curva.predict_proba(extremos)[:, 1].round(2))
assert recta_h.predict(extremos).min() < 0 or recta_h.predict(extremos).max() > 1
assert ((curva.predict_proba(extremos)[:, 1] > 0) & (curva.predict_proba(extremos)[:, 1] < 1)).all()
print(f"\nCada unidad mas de lluvia multiplica por {np.exp(curva.coef_[0][0]):.1f} las 'posibilidades' (odds) de que aparezca el hongo")

---
**Lección:** con lluvia extrema la recta da «probabilidades» **fuera de 0 a 1** (sin sentido), mientras la curva en «S» siempre da valores válidos. Las «posibilidades» (*odds*: probabilidad de sí dividida entre probabilidad de no) se **multiplican** con cada unidad de lluvia; ojo: eso **no** es «el hongo es X veces más probable», es una razón entre posibilidades.

---
## 3. El Vendedor Tramposo: Promedio contra Mediana 🧮

En el mercado se anotan precios por bulto de papa para 40 bultos de distinto peso (**inventados**). Pero un vendedor anotó un precio disparatado. El **promedio** se deja arrastrar; la **mediana** (el valor del medio) casi ni se entera. Lo mismo pasa con las rectas: la recta de siempre es «de promedios» y se tuerce; la **recta de la mediana** resiste.

In [ ]:
rng = np.random.default_rng(5)
peso = rng.uniform(40, 60, 40)                                  # kg por bulto
precio = 2 * peso - 20 + rng.normal(0, 3, 40)                   # precio "honesto": sube 2 por cada kg (miles de pesos, INVENTADO)
precio_malo = precio.copy()
precio_malo[np.argmax(peso)] += 400                             # un vendedor anoto 400 de mas

print(f"Promedio del precio: sin el dato loco = {precio.mean():.1f} | con el dato loco = {precio_malo.mean():.1f}")
print(f"Mediana del precio:  sin el dato loco = {np.median(precio):.1f} | con el dato loco = {np.median(precio_malo):.1f}")
assert precio_malo.mean() - precio.mean() > 9 and abs(np.median(precio_malo) - np.median(precio)) < 1.5

XP = peso.reshape(-1, 1)
rec_buena = LinearRegression().fit(XP, precio).coef_[0]
rec_mala = LinearRegression().fit(XP, precio_malo).coef_[0]
rec_med = QuantileRegressor(quantile=0.5, alpha=0, solver="highs").fit(XP, precio_malo).coef_[0]
print(f"\nCuanto sube el precio por cada kg (verdad = 2.0): recta sin dato loco = {rec_buena:.2f} | recta CON dato loco = {rec_mala:.2f} | recta de la mediana CON dato loco = {rec_med:.2f}")
assert abs(rec_mala - 2) > 0.5 and abs(rec_med - 2) < 0.4

---
##### 🛠️ Práctica 1: Tu turno: ¿cuántas plagas con humedad muy alta?

Con el modelo de conteos `poisson` de la sección 1 (ya ajustado con `humedad`):

1. Calcula `pred_alta = poisson.predict([[2.0]])[0]` (plagas esperadas con humedad = 2) y `pred_baja = poisson.predict([[-2.0]])[0]`.
2. Calcula `veces = pred_alta / pred_baja`.
3. Verifica con `assert` que `pred_alta > pred_baja > 0` y que `veces > 5`. Comenta con tus palabras: ¿por qué la recta no podría dar una relación «de veces» así de clara?

In [ ]:
# Escribe tu código aquí

# pred_alta = ...
# pred_baja = ...
# veces = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
pred_alta = poisson.predict([[2.0]])[0]
pred_baja = poisson.predict([[-2.0]])[0]
veces = pred_alta / pred_baja
print(f"humedad 2: {pred_alta:.1f} plagas | humedad -2: {pred_baja:.2f} plagas | {veces:.1f} veces mas")

assert pred_alta > pred_baja > 0 and veces > 5
```
</details>

---


---
**Lección:** un solo dato loco subió el promedio en 10 mil pesos (de 79.5 a 89.5) y torció la recta (de 2.03 a 4.72 mil por kg), pero **la mediana y la recta de la mediana casi no se movieron**. Eso se llama ser **robusto**. El precio de la robustez: si **no** hay datos locos, el promedio es un poco más preciso; por eso se usa la recta robusta **cuando hay motivos para desconfiar de los datos**, no siempre.

---
## Resumen en una frase 🎯

> **La línea recta se porta mal con conteos (predice negativos), con sí/no (se sale de 0 a 1) y con datos locos (se tuerce); sus primos —el modelo de conteos, la curva en «S» y la recta de la mediana— respetan lo que se mide y resisten a los atípicos.**

### Lo que aprendiste hoy:
- ✅ Los **conteos** nunca son negativos: se modelan con una forma **multiplicativa** («cada punto de humedad multiplica las plagas por 2.2»).
- ✅ Un **sí/no** se modela con una **curva en «S»** que se queda entre 0 y 1; sus efectos se leen como «posibilidades» (*odds*), no como «veces más probable».
- ✅ El **promedio** se deja arrastrar por un dato loco; la **mediana** casi no.
- ✅ La **recta de la mediana** es robusta, pero pierde un poco de precisión cuando los datos son limpios.

---
### 🧠 ¿Ya lo tienes claro?

1. Una recta predice –3 pollos para una granja pequeña. ¿Qué te dice eso del modelo? ¿Qué tipo de modelo usarías para contar pollos?
2. Si el modelo dice que la lluvia «multiplica por 6 las posibilidades» de un hongo, ¿significa que el hongo es 6 veces más probable? Explica con tus palabras.
3. Tienes precios de 30 puestos del mercado y uno de ellos tiene un error de dedo. ¿Reportarías el promedio o la mediana? ¿Por qué?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>